# DQL: Basic Operators

Operators build filter conditions. This notebook shows common SQL operators and their PySpark equivalents.


## Setup

Run this first. It creates a Spark session, finds the CSV files, loads them as DataFrames, and registers SQL temp views.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd(),
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or beside this notebook.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Equality and Inequality

Use `==` for equality in the DataFrame API. Use `!=` for not equal.


In [ ]:
emp.where(F.col("job") == "manager").select("empno", "ename", "job").show()
emp.where(F.col("job") != "manager").select("empno", "ename", "job").show(10)

spark.sql("""
SELECT empno, ename, job FROM emp WHERE job = 'manager'
""").show()

spark.sql("""
SELECT empno, ename, job FROM emp WHERE job <> 'manager' LIMIT 10
""").show()


## IN and NOT IN

Use `isin` for membership checks.


In [ ]:
emp.where(F.col("deptno").isin(10, 30)).select("empno", "ename", "deptno").show()
emp.where(~F.col("deptno").isin(10, 30)).select("empno", "ename", "deptno").show(10)

spark.sql("""
SELECT empno, ename, deptno FROM emp WHERE deptno IN (10, 30)
""").show()

spark.sql("""
SELECT empno, ename, deptno FROM emp WHERE deptno NOT IN (10, 30) LIMIT 10
""").show()


## LIKE and NOT LIKE

Use `like` for wildcard matching. `%` means any number of characters.


In [ ]:
emp.where(F.col("ename").like("a%")).select("empno", "ename").show()
emp.where(~F.col("ename").like("%_1%")).select("empno", "ename").show(10)

spark.sql("""
SELECT empno, ename FROM emp WHERE ename LIKE 'a%'
""").show()

spark.sql("""
SELECT empno, ename FROM emp WHERE ename NOT LIKE '%_1%' LIMIT 10
""").show()


## BETWEEN

`between` includes both endpoints.


In [ ]:
emp.where(F.col("sal").between(2000, 3000)).select("empno", "ename", "sal").orderBy("sal").show()

spark.sql("""
SELECT empno, ename, sal
FROM emp
WHERE sal BETWEEN 2000 AND 3000
ORDER BY sal
""").show()


## Comparison Operators

Use greater than, greater than or equal, less than, and less than or equal for numeric and date comparisons.


In [ ]:
emp.select("empno", "ename", "sal").where(F.col("sal") > 3000).show()
emp.select("empno", "ename", "sal").where(F.col("sal") >= 3000).show()
emp.select("empno", "ename", "sal").where(F.col("sal") < 1200).show()
emp.select("empno", "ename", "sal").where(F.col("sal") <= 1200).show()

spark.sql("""
SELECT empno, ename, sal FROM emp WHERE sal > 3000
""").show()


## EXISTS and NOT EXISTS

Spark SQL supports correlated subqueries with `EXISTS`. In the DataFrame API, semi and anti joins are common equivalents.


In [ ]:
emp.join(emp_projects.select("empno").distinct(), on="empno", how="left_semi").select("empno", "ename").show(10)
emp.join(emp_projects.select("empno").distinct(), on="empno", how="left_anti").select("empno", "ename").show(10)

spark.sql("""
SELECT e.empno, e.ename
FROM emp e
WHERE EXISTS (
    SELECT 1 FROM emp_projects ep WHERE ep.empno = e.empno
)
ORDER BY e.empno
LIMIT 10
""").show()

spark.sql("""
SELECT e.empno, e.ename
FROM emp e
WHERE NOT EXISTS (
    SELECT 1 FROM emp_projects ep WHERE ep.empno = e.empno
)
ORDER BY e.empno
LIMIT 10
""").show()
